Initialise ifcopenshell libraries and conditions

In [841]:
import numpy as np
import ifcopenshell

# for some reason ifcopenshell.api is not imported when importing ifcopenshell, so we need to import it explicitly
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.file()

# create the root entity (IfcProject) and set it as the root of the model
ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject")

# set coordinates to be in meters
length = ifcopenshell.api.unit.add_si_unit(model, unit_type="LENGTHUNIT")
ifcopenshell.api.unit.assign_unit(model, units=[length])

# create contexts for 3D model, plan view and body
model3d = ifcopenshell.api.context.add_context(model, context_type="Model")
plan = ifcopenshell.api.context.add_context(model, context_type="Plan")
body = ifcopenshell.api.context.add_context(model, 
    context_type="Model", context_identifier="Body", target_view="MODEL_VIEW", parent=model3d)

# add some contexts for axes and annotations
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Axis", target_view="GRAPH_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Axis", target_view="GRAPH_VIEW", parent=plan)
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Box", target_view="MODEL_VIEW", parent=model3d)
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Annotation", target_view="PLAN_VIEW", parent=plan)


# create the project structure components for use in IfcRelAggregates
project = model.by_type("IfcProject")[0]
# site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
# building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
site = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSite", name="Main Site")
building = ifcopenshell.api.root.create_entity(model, ifc_class="IfcBuilding", name="Building")
storey = model.create_entity("IfcBuildingStorey", Name="Ground", Elevation=0.0)

# IfcRelAggregates is used to define the hierarchical structure of the building model.
model.create_entity("IfcRelAggregates", RelatingObject=project, RelatedObjects=[site])
model.create_entity("IfcRelAggregates", RelatingObject=site, RelatedObjects=[building])
model.create_entity("IfcRelAggregates", RelatingObject=building, RelatedObjects=[storey])


#23=IfcRelAggregates($,$,$,$,#19,(#20))

Create function to create IfcSpaces, Walls, Floors

In [842]:
import profile

def create_space(length, width, name, origin, height = 3.0):
    
    # create the specific room ifcspace and relate it to the storey in the model hierarchy
    ifcspace = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace", name=name)
    model.create_entity("IfcRelAggregates", RelatingObject=storey, RelatedObjects=[ifcspace])

    # create the actual space entity with geometry and relate it to the ifcspace in the model hierarchy
    room_space = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSpace", name=name)
    # model.create_entity("IfcRelAggregates", RelatingObject=storey, RelatedObjects=[space])

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=room_space, matrix=matrix, is_si=False)

    profile = model.create_entity("IfcRectangleProfileDef",
        ProfileName=f"{name} footprint", ProfileType="AREA", 
        XDim=length, YDim=width)
    
    space_rep = ifcopenshell.api.geometry.add_profile_representation(
        model, context=body, profile=profile, depth=height)
    
    model.create_entity("IfcRelContainedInSpatialStructure", RelatingStructure=ifcspace, RelatedElements=[room_space])

    return ifcspace, room_space


# define a function to add ifcwall to the model
def create_wall(length, name: str, origin, rotate, ifcspace):
    wall = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall", name=name)
    
    wall_rep = ifcopenshell.api.geometry.add_wall_representation(model,
        context=body, length=length, height=3.0, thickness=0.2)
    
    matrix = np.eye(4)
    if rotate:
        matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
    matrix[:,3][0:3] = origin
    
    ifcopenshell.api.geometry.edit_object_placement(model, product=wall, matrix=matrix, is_si=False)
    ifcopenshell.api.geometry.assign_representation(model, product=wall, representation=wall_rep)

    # this part is important because it relates the wall back to the hierarchy of the ifc model defined before
    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace, products=[wall])

    return wall

def create_slab(length, width, name, origin, ifcspace):
    slab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab", name=name)

    # Create a rectangular profile for the slab with your desired dimensions
    profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
        XDim=length, YDim=width)

    slab_rep = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

    matrix = np.eye(4)
    matrix[:,3][0:3] = origin
    ifcopenshell.api.geometry.edit_object_placement(model, product=slab, matrix=matrix, is_si=True)
    ifcopenshell.api.geometry.assign_representation(model, product=slab, representation=slab_rep)
    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace, products=[slab])
    
    return slab


# define a function to add a room to the model, which is just an ifcspace with physical geometry assigned
def create_room(length, width, name, origin):
    
    room_dict = {}

    [ifcspace, space] = create_space(length=length, width=width, name=name, origin=origin+np.array([length/2, width/2, 0]))

    room_dict[name+"wallN"] = create_wall(length=length, name=name+"wallN", origin=origin+np.array([0, width, 0]), rotate=False, ifcspace=ifcspace)
    room_dict[name+"wallS"] = create_wall(length=length, name=name+"wallS", origin=origin+np.array([0,-0.2,0]), rotate=False, ifcspace=ifcspace)

    room_dict[name+"wallE"] = create_wall(length=width+0.4, name=name+"wallE", origin=origin+np.array([length+0.2, -0.2, 0]), rotate=True, ifcspace=ifcspace)
    room_dict[name+"wallW"] = create_wall(length=width+0.4, name=name+"wallW", origin=origin+np.array([0, -0.2, 0]), rotate=True, ifcspace=ifcspace)

    room_dict[name+"slab"] = create_slab(length=length+0.4, width=width+0.4, name=name+"slab", origin=origin+np.array([length/2, width/2, -0.2]), ifcspace=ifcspace)   
    
    return [space, room_dict]

In [843]:
# Define a function to add space boundaries to the model. 
# This function creates an IfcRelSpaceBoundary entity, relates it to the space and the building element, 
# and sets the physical and internal/external attributes.

def create_rel(space, element, physical="VIRTUAL", internal_external="INTERNAL", name=None):
    rel = ifcopenshell.api.root.create_entity(
        model,
        ifc_class="IfcRelSpaceBoundary2ndLevel",
        name=name
    )
    ifcopenshell.api.boundary.edit_attributes(
        model,
        entity=rel,
        relating_space=space,
        related_building_element=element,
    )
    rel.PhysicalOrVirtualBoundary = physical
    rel.InternalOrExternalBoundary = internal_external
    return rel

Function for create openings

In [844]:
# adds openings, specifically for doors
# for now, this requires knowledge of both the walls of both spaces, including names. Can be tedious

# this currently creates two openings if there are two walls, but they are both related to the same opening element for simplicity. 
# This is because the opening element is what relates the two spaces together, and the geometry is just for visualisation purposes.
# this does not mean that the extra opening is a "leftover" element within the IFC model. It exists purely as a physical geometrical visualisation. It is not ever represented as relationship to an IfcSpace.
# openings are essentially viewed as subtractions to the wall.
# as a matter of fact, you cannot simply (or at least conventionally) define a relationship between an opening and any other object directly. However they can be defined as the link between two IfcSpace entities.
def create_opening(width, height, name, origin, ifcspace1, wall1, ifcspace2=None, wall2=None):
    opening = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement", name=name)
    opening_rep = ifcopenshell.api.geometry.add_wall_representation(model,
        context=body, length=width, height=height, thickness=0.4)
    ifcopenshell.api.geometry.assign_representation(model, product=opening, representation=opening_rep)

    matrix = np.eye(4)
    if wall1.Name[-1] == "W" or wall1.Name[-1] == "E":
        matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
    matrix[:,3][0:3] = origin

    ifcopenshell.api.geometry.edit_object_placement(model, product=opening, matrix=matrix, is_si=True)

    # issue. I suspect that this will need to select both walls that the opening intersects with
    # i think it's fine, because technically i just need to know which space the opening connects to.
    # plus i am manually placing the openings. it's more for geometry and visuals
    ifcopenshell.api.feature.add_feature(model, feature=opening, element=wall1)
    ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace1, products=[opening])


    if (wall2 is not None):
        opening2 = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement", name=name)
        opening2_rep = ifcopenshell.api.geometry.add_wall_representation(model,
            context=body, length=width, height=height, thickness=0.4)
        ifcopenshell.api.geometry.assign_representation(model, product=opening2, representation=opening2_rep)

        ifcopenshell.api.geometry.edit_object_placement(model, product=opening2, matrix=matrix, is_si=True)

        ifcopenshell.api.feature.add_feature(model, feature=opening2, element=wall2)
        ifcopenshell.api.spatial.assign_container(model, relating_structure=ifcspace2, products=[opening2])
        # b2=create_rel(ifcspace2, opening2, name=name+"_Boundary")
        
        # create two way relationship entities between two rooms
        b1=create_rel(ifcspace1, opening, name=name+"_Boundary")
        b2=create_rel(ifcspace2, opening, name=name+"_Boundary")

        # the below is the same as create_space_boundary but with an added element
        ifcopenshell.api.boundary.edit_attributes(
            model,
            entity=b1,
            relating_space=ifcspace1,
            related_building_element=opening,
            corresponding_boundary=b2
        )

        ifcopenshell.api.boundary.edit_attributes(
            model,
            entity=b2,
            relating_space=ifcspace2,
            related_building_element=opening, # this is intentional, so b2 can be related to b1 through the same opening. openign2 is just for geometry visualisation
            corresponding_boundary=b1
        )
    return opening

Functions for extracting information from model

In [845]:
# defined as a dictionary, because there can be multiple boundaries related by openings
def get_boundary_room(space):
    b_room = {}
    for boundary in space.BoundedBy:
        corr_boundary = boundary.CorrespondingBoundary
        b_room[boundary.RelatedBuildingElement.Name] = corr_boundary.RelatingSpace
    return b_room

def get_openings(space):
    openings = {}
    for boundary in space.BoundedBy:
        if boundary.RelatedBuildingElement.is_a("IfcOpeningElement"):
            openings[boundary.RelatedBuildingElement.Name] = boundary.RelatedBuildingElement
    return openings

def get_coordinates(entity):
    matrix = ifcopenshell.util.placement.get_local_placement(entity.ObjectPlacement)
    return matrix[:,3][0:3]

###### other functions to add later:
# get length/width of a space
# calculate distance between spaces

Model Generation Code

In [846]:
# a = create_room(length=5, width=4, name="Room1", origin=(0,0,0))
# # when adding rooms next to each other, make sure to account for wall thickness so that they are flush with each other.
# # there is also way to way to account for shared walls between rooms, but for simplicity we will just add separate walls
# # for each room and make sure they are flush with each other
# b = create_room(length=3, width=4, name="Room2", origin=(5.2,0,0))
# c = create_room(length=8.2, width=2, name="Corridor", origin=(0,4.2,0))

# # Define rooms in order. Preferably from inside out
# # d=create_opening(width=1, height=2, name="Room1_Door", origin=(2, 4, 0), ifcspace1=a[0], wall1=a[1]["Room1wallN"], ifcspace2=c[0], wall2=c[1]["CorridorwallS"])
# d=create_opening(width=1, height=2, name="Room1_Door", origin=(2, 4, 0), ifcspace1=a[0], wall1=a[1]["Room1wallN"], ifcspace2=c[0], wall2=c[1]["CorridorwallS"])


# entry_lobby = create_room(length=6, width=8, name="entry_lobby", origin=(0,6,0))
# s1_office = create_room(length=6, width=6, name="s1_office", origin=(6.2,3,0))
# s2_office = create_room(length=6, width=6, name="s2_office", origin=(12.4,3,0))
# s3_office = create_room(length=6, width=6, name="s3_office", origin=(18.6,3,0))
# east_lobby = create_room(length=4, width=9, name="east_lobby", origin=(24.8,3,0))
# west_corridor = create_room(length=10, width=2.8, name="west_corridor", origin=(6.2,9.2,0))
# east_corridor = create_room(length=8.2, width=2.8, name="east_corridor", origin=(16.4,9.2,0))
# n1_office = create_room(length=3, width=6, name="n1_office", origin=(6.2,12.2,0))
# n2_office = create_room(length=4, width=6, name="n2_office", origin=(9.4,12.2,0))
# north_corridor = create_room(length=2, width=6, name="north_corridor", origin=(13.6,12.2,0))
# north_store = create_room(length=2, width=6, name="north_store", origin=(15.8,12.2,0))
# n3_office = create_room(length=4, width=6, name="n3_office", origin=(18,12.2,0))
# ne_lobby = create_room(length=6.6, width=6, name="ne_lobby", origin=(22.2,12.2,0))
# north_clean = create_room(length=4.2, width=3, name="north_clean", origin=(13.6,18.4,0))


######## SW Block (17.2 x 23.2)
OR2 = create_room(length=8.6, width=9.5, name="OR2", origin=(0,0,0))
AB2 = create_room(length=5.0, width=4.3, name="AB2", origin=(8.8,0,0))
ExB2 = create_room(length=5.0, width=4.3, name="ExB2", origin=(8.8,4.5,0))
Setup1 = create_room(length=8.6, width=3.6, name="Setup1", origin=(0,9.7,0))
OR1 = create_room(length=8.6, width=9.5, name="OR1", origin=(0,13.5,0))
Set_Corr = create_room(length=5.0, width=3.2, name="Set_Corr", origin=(8.8,9.0,0))
Store1 = create_room(length=5.0, width=2.5, name="Store1", origin=(8.8,12.4,0))
ExB1 = create_room(length=5.0, width=3.5, name="ExB1", origin=(8.8,15.1,0))
AB1 = create_room(length=5.0, width=4.2, name="AB1", origin=(8.8,18.8,0))
SW_Circ = create_room(length=3.0, width=23, name="SW_Circ", origin=(14,0,0))

########### S Block (41.7 x 23.2)
ST_Stock = create_room(length=8.7, width=12.2, name="ST_Stock", origin=(17.2,0,0))
Cath_Lab = create_room(length=9.0, width=9.3, name="Cath_Lab", origin=(26.1,0,0))
ExB3 = create_room(length=4.2, width=3.5, name="ExB3", origin=(17.2,12.4,0))
AB3 = create_room(length=4.2, width=4.0, name="AB3", origin=(17.2,16.1,0))
Hyb_Theatre = create_room(length=13.5, width=7.7, name="Hyb_Theatre", origin=(21.6,12.4,0))
IT1 = create_room(length=4.2, width=2.7, name="IT1", origin=(17.2,20.3,0))
Med1 = create_room(length=4.0, width=2.7, name="Med1", origin=(24.6,20.3,0))
Ctrl1 = create_room(length=6.5, width=2.7, name="Ctrl1", origin=(28.6,20.3,0))
Lab1 = create_room(length=3.0, width=6.3, name="Lab1", origin=(35.3,0,0))
Scrub1 = create_room(length=3.0, width=5.3, name="Scrub1", origin=(35.3,6.5,0))
CPU2 = create_room(length=3.0, width=3.7, name="CPU2", origin=(35.3,12.0,0))
Store2 = create_room(length=3.0, width=3.3, name="Store2", origin=(35.3,15.9,0))
CPU1 = create_room(length=3.0, width=3.6, name="CPU1", origin=(35.3,19.4,0))
SE_Circ = create_room(length=3.0, width=23.0, name="SE_Circ", origin=(38.5,0,0))


############ SE Block
ST_Equip1 = create_room(length=9.5, width=3.0, name="ST_Equip1", origin=(44.7,0,0))
Off1 = create_room(length=3.7, width=3.0, name="Off1", origin=(54.4,0,0))
Off2 = create_room(length=3.8, width=3.0, name="Off2", origin=(60.2,0,0))
AmenitiesF = create_room(length=12.0, width=3.0, name="AmenitiesF", origin=(64.2,0,0))
Travel1 = create_room(length=24, width=2.2, name="Travel1", origin=(41.7,3.2,0))

Disp1 = create_room(length=3.2, width=2.6, name="Disp1", origin=(41.7,5.6,0))
Clean2 = create_room(length=3.2, width=10.6, name="Clean2", origin=(41.7,8.4,0))
WC = create_room(length=3.2, width=3.8, name="WC", origin=(41.7,19.2,0))

LiftLobby1 = create_room(length=4.7, width=8.5, name="LiftLobby1", origin=(45.1,5.6,0))
Lift_Circ = create_room(length=3.2, width=8.7, name="Lift_Circ", origin=(45.1,14.3,0))

LiftLobby2 = create_room(length=4.0, width=10.6, name="LiftLobby2", origin=(57.3,5.6,0))
AmenitiesM = create_room(length=12.0, width=4.0, name="AmenitiesM", origin=(64.2,5.6,0))

Travel2 = create_room(length=14.7, width=2.2, name="Travel2", origin=(61.5,14,0))
Int1 = create_room(length=1.6, width=4.0, name="Int1", origin=(61.5,9.8,0))
Int2 = create_room(length=2.7, width=4.0, name="Int2", origin=(63.3,9.8,0))
Int3 = create_room(length=2.8, width=4.0, name="Int3", origin=(66.2,9.8,0))
Int4 = create_room(length=2.7, width=4.0, name="Int4", origin=(69.2,9.8,0))
Cons1 = create_room(length=4.1, width=4.0, name="Cons1", origin=(72.1,9.8,0))

Wait = create_room(length=5.2, width=6.6, name="Wait", origin=(57.3,16.4,0))
Copy1 = create_room(length=4.9, width=6.6, name="Copy1", origin=(62.7,16.4,0))
Change_Corr = create_room(length=2.1, width=6.6, name="Change_Corr", origin=(67.8,16.4,0))
Lounge = create_room(length=3.3, width=5.3, name="Lounge", origin=(72.9,16.4,0))

############# NW Block (37.4)
ST_Gen = create_room(length=4.7, width=2.9, name="ST_Gen", origin=(-1.0,23.2,0))
Meet1 = create_room(length=4.7, width=3.3, name="Meet1", origin=(-1.0,26.3,0))
StaffR = create_room(length=4.7, width=7.4, name="StaffR", origin=(-1.0,29.8,0))

NW_Circ = create_room(length=41.0, width=2.9, name="NW_Circ", origin=(3.9,23.2,0))

Stair_Circ = create_room(length=2.3, width=4.8, name="Stair_Circ", origin=(3.9,26.3,0))
ExB4 = create_room(length=4.3, width=4.1, name="ExB4", origin=(8.6,26.3,0))
AnBay1 = create_room(length=4.9, width=4.1, name="AnBay1", origin=(13.1,26.3,0))
Setup2 = create_room(length=4.8, width=4.1, name="Setup2", origin=(18.2,26.3,0))
BayCirc1 = create_room(length=2.2, width=6.5, name="BayCirc1", origin=(23.2,26.3,0))
ST_Equip2 = create_room(length=3.3, width=4.1, name="ST_Equip2", origin=(25.6,26.3,0))

OR3 = create_room(length=11, width=6.8, name="OR3", origin=(8.6,30.6,0))
Clean1 = create_room(length=3.2, width=2.2, name="Clean1", origin=(19.8,30.6,0))
Dirty1 = create_room(length=5.6, width=4.4, name="Dirty1", origin=(19.8,33,0))

######### N Block
ST_Equip3 = create_room(length=3.2, width=4.1, name="ST_Equip3", origin=(34.8,26.3,0))
Clean3 = create_room(length=2.9, width=4.1, name="Clean3", origin=(38.2,26.3,0))
Bay_Res = create_room(length=6.4, width=4.1, name="Bay_Res", origin=(41.3,26.3,0))

RecBay_Circ1 = create_room(length=39.5, width=6.8, name="RecBay_Circ1", origin=(25.6,30.6,0))
Dirty2 = create_room(length=5.7, width=2.8, name="Dirty2", origin=(56.7,27.6,0))

NE_Circ = create_room(length=24.8, width=2.9, name="NE_Circ", origin=(45.1,23.2,0))

ACC1 = create_room(length=4.6, width=2.8, name="ACC1", origin=(65.3,27.6,0))
RecBay_Circ3 = create_room(length=19.8, width=8.5, name="RecBay_Circ3", origin=(70.1,21.9,0))
RecBay_Circ2 = create_room(length=24.6, width=6.8, name="RecBay_Circ2", origin=(65.3,30.6,0))
Plant = create_room(length=9.0, width=8.5, name="Plant", origin=(90.1,21.9,0))




In [847]:
OR2_AB2 = create_opening(width=2, height=2, name="OR2_AB2", origin=(8.8, 2.0, 0), ifcspace1=OR2[0], wall1=OR2[1]["OR2wallE"], ifcspace2=AB2[0], wall2=AB2[1]["AB2wallW"])
OR2_ExB2 = create_opening(width=2, height=2, name="OR2_ExB2", origin=(8.8, 5.0, 0), ifcspace1=OR2[0], wall1=OR2[1]["OR2wallE"], ifcspace2=ExB2[0], wall2=ExB2[1]["ExB2wallW"])

AB2_SW_Circ = create_opening(width=2, height=2, name="AB2_SW_Circ", origin=(14, 2.0, 0), ifcspace1=AB2[0], wall1=AB2[1]["AB2wallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])
ExB2_SW_Circ = create_opening(width=2, height=2, name="ExB2_SW_Circ", origin=(14, 5.0, 0), ifcspace1=ExB2[0], wall1=ExB2[1]["ExB2wallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])

OR2_Setup1 = create_opening(width=1, height=2, name="OR2_Setup1", origin=(5.0, 9.5, 0), ifcspace1=OR2[0], wall1=OR2[1]["OR2wallN"], ifcspace2=Setup1[0], wall2=Setup1[1]["Setup1wallS"])
Setup1_OR1 = create_opening(width=1, height=2, name="Setup1_OR1", origin=(5.0, 13.3, 0), ifcspace1=Setup1[0], wall1=Setup1[1]["Setup1wallN"], ifcspace2=OR1[0], wall2=OR1[1]["OR1wallS"])

Setup1_Set_Corr = create_opening(width=1, height=2, name="Setup1_Set_Corr", origin=(8.8, 10.0, 0), ifcspace1=Setup1[0], wall1=Setup1[1]["Setup1wallE"], ifcspace2=Set_Corr[0], wall2=Set_Corr[1]["Set_CorrwallW"])
Set_Corr_SW_Circ = create_opening(width=1, height=2, name="Set_Corr_SW_Circ", origin=(14.0, 10.0, 0), ifcspace1=Set_Corr[0], wall1=Set_Corr[1]["Set_CorrwallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])

Store1_SW_Circ = create_opening(width=1, height=2, name="Store1_SW_Circ", origin=(14.0, 13.0, 0), ifcspace1=Store1[0], wall1=Store1[1]["Store1wallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])

OR1_ExB1 = create_opening(width=2, height=2, name="OR1_ExB1", origin=(8.8, 16.0, 0), ifcspace1=OR1[0], wall1=OR1[1]["OR1wallE"], ifcspace2=ExB1[0], wall2=ExB1[1]["ExB1wallW"])
ExB1_SW_Circ = create_opening(width=2, height=2, name="ExB1_SW_Circ", origin=(14.0, 16.0, 0), ifcspace1=ExB1[0], wall1=ExB1[1]["ExB1wallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])

OR1_AB1 = create_opening(width=2, height=2, name="OR1_AB1", origin=(8.8, 19.5, 0), ifcspace1=OR1[0], wall1=OR1[1]["OR1wallE"], ifcspace2=AB1[0], wall2=AB1[1]["AB1wallW"])
AB1_SW_Circ = create_opening(width=2, height=2, name="AB1_SW_Circ", origin=(14.0, 19.5, 0), ifcspace1=AB1[0], wall1=AB1[1]["AB1wallE"], ifcspace2=SW_Circ[0], wall2=SW_Circ[1]["SW_CircwallW"])

SW_Circ_NW_Circ = create_opening(width=3, height=2, name="SW_Circ_NW_Circ", origin=(14, 23, 0), ifcspace1=SW_Circ[0], wall1=SW_Circ[1]["SW_CircwallN"], ifcspace2=NW_Circ[0], wall2=NW_Circ[1]["NW_CircwallS"])

SW_Circ_ST_Stock = create_opening(width=1, height=2, name="SW_Circ_ST_Stock", origin=(17.2, 10, 0), ifcspace1=SW_Circ[0], wall1=SW_Circ[1]["SW_CircwallE"], ifcspace2=ST_Stock[0], wall2=ST_Stock[1]["ST_StockwallW"])

SW_Circ_ExB3 = create_opening(width=2, height=2, name="SW_Circ_ExB3", origin=(17.2, 13.2, 0), ifcspace1=SW_Circ[0], wall1=SW_Circ[1]["SW_CircwallE"], ifcspace2=ExB3[0], wall2=ExB3[1]["ExB3wallW"])
SW_Circ_AB3 = create_opening(width=2, height=2, name="SW_Circ_AB3", origin=(17.2, 17.0, 0), ifcspace1=SW_Circ[0], wall1=SW_Circ[1]["SW_CircwallE"], ifcspace2=AB3[0], wall2=AB3[1]["AB3wallW"])
SW_Circ_IT1 = create_opening(width=1, height=2, name="SW_Circ_IT1", origin=(17.2, 20.5, 0), ifcspace1=SW_Circ[0], wall1=SW_Circ[1]["SW_CircwallE"], ifcspace2=IT1[0], wall2=IT1[1]["IT1wallW"])

ExB3_Hyb_Theatre = create_opening(width=2, height=2, name="ExB3_Hyb_Theatre", origin=(21.6, 13.2, 0), ifcspace1=ExB3[0], wall1=ExB3[1]["ExB3wallE"], ifcspace2=Hyb_Theatre[0], wall2=Hyb_Theatre[1]["Hyb_TheatrewallW"])
AB3_Hyb_Theatre = create_opening(width=2, height=2, name="AB3_Hyb_Theatre", origin=(21.6, 17.0, 0), ifcspace1=AB3[0], wall1=AB3[1]["AB3wallE"], ifcspace2=Hyb_Theatre[0], wall2=Hyb_Theatre[1]["Hyb_TheatrewallW"])

Hyb_Theatre_Ctrl1 = create_opening(width=1, height=2, name="Hyb_Theatre_Ctrl1", origin=(29.0, 20.1, 0), ifcspace1=Hyb_Theatre[0], wall1=Hyb_Theatre[1]["Hyb_TheatrewallN"], ifcspace2=Ctrl1[0], wall2=Ctrl1[1]["Ctrl1wallS"])
Ctrl1_NW_Circ = create_opening(width=1, height=2, name="Ctrl1_NW_Circ", origin=(29.0, 23.0, 0), ifcspace1=Ctrl1[0], wall1=Ctrl1[1]["Ctrl1wallN"], ifcspace2=NW_Circ[0], wall2=NW_Circ[1]["NW_CircwallS"])
Med1_NW_Circ = create_opening(width=1, height=2, name="Med1_NW_Circ", origin=(27.2, 23.0, 0), ifcspace1=Med1[0], wall1=Med1[1]["Med1wallN"], ifcspace2=NW_Circ[0], wall2=NW_Circ[1]["NW_CircwallS"])


Cath_Lab_Scrub1 = create_opening(width=1, height=2, name="Cath_Lab_Scrub1", origin=(35.3, 7.5, 0), ifcspace1=Cath_Lab[0], wall1=Cath_Lab[1]["Cath_LabwallE"], ifcspace2=Scrub1[0], wall2=Scrub1[1]["Scrub1wallW"])

Lab1_Scrub1 = create_opening(width=1, height=2, name="Lab1_Scrub1", origin=(36.8, 6.3, 0), ifcspace1=Lab1[0], wall1=Lab1[1]["Lab1wallN"], ifcspace2=Scrub1[0], wall2=Scrub1[1]["Scrub1wallS"])

Scrub1_SE_Circ = create_opening(width=1, height=2, name="Scrub1_SE_Circ", origin=(38.5, 7.5, 0), ifcspace1=Scrub1[0], wall1=Scrub1[1]["Scrub1wallE"], ifcspace2=SE_Circ[0], wall2=SE_Circ[1]["SE_CircwallW"])
CPU2_SE_Circ = create_opening(width=1, height=2, name="CPU2_SE_Circ", origin=(38.5, 12.2, 0), ifcspace1=CPU2[0], wall1=CPU2[1]["CPU2wallE"], ifcspace2=SE_Circ[0], wall2=SE_Circ[1]["SE_CircwallW"])
Store2_SE_Circ = create_opening(width=1, height=2, name="Store2_SE_Circ", origin=(38.5, 16.9, 0), ifcspace1=Store2[0], wall1=Store2[1]["Store2wallE"], ifcspace2=SE_Circ[0], wall2=SE_Circ[1]["SE_CircwallW"])
CPU1_SE_Circ = create_opening(width=1, height=2, name="CPU1_SE_Circ", origin=(38.5, 19.6, 0), ifcspace1=CPU1[0], wall1=CPU1[1]["CPU1wallE"], ifcspace2=SE_Circ[0], wall2=SE_Circ[1]["SE_CircwallW"])

SE_Circ_NW_Circ = create_opening(width=3, height=2, name="SE_Circ_NW_Circ", origin=(38.5, 23, 0), ifcspace1=SE_Circ[0], wall1=SE_Circ[1]["SE_CircwallN"], ifcspace2=NW_Circ[0], wall2=NW_Circ[1]["NW_CircwallS"])

SE_Circ_Travel1 = create_opening(width=1, height=2, name="SE_Circ_Travel1", origin=(41.7, 4.0, 0), ifcspace1=SE_Circ[0], wall1=SE_Circ[1]["SE_CircwallE"], ifcspace2=Travel1[0], wall2=Travel1[1]["Travel1wallW"])
SE_Circ_Disp1 = create_opening(width=1, height=2, name="SE_Circ_Disp1", origin=(41.7, 6.5, 0), ifcspace1=SE_Circ[0], wall1=SE_Circ[1]["SE_CircwallE"], ifcspace2=Disp1[0], wall2=Disp1[1]["Disp1wallW"])
SE_Circ_Clean2 = create_opening(width=1, height=2, name="SE_Circ_Clean2", origin=(41.7, 13.5, 0), ifcspace1=SE_Circ[0], wall1=SE_Circ[1]["SE_CircwallE"], ifcspace2=Clean2[0], wall2=Clean2[1]["Clean2wallW"])

ST_Equip1_Travel1 = create_opening(width=1, height=2, name="ST_Equip1_Travel1", origin=(46.2, 3.0, 0), ifcspace1=ST_Equip1[0], wall1=ST_Equip1[1]["ST_Equip1wallN"], ifcspace2=Travel1[0], wall2=Travel1[1]["Travel1wallS"])
Off1_Travel1 = create_opening(width=1, height=2, name="Off1_Travel1", origin=(54.6, 3.0, 0), ifcspace1=Off1[0], wall1=Off1[1]["Off1wallN"], ifcspace2=Travel1[0], wall2=Travel1[1]["Travel1wallS"])
Off2_Travel1 = create_opening(width=1, height=2, name="Off2_Travel1", origin=(62.8, 3.0, 0), ifcspace1=Off2[0], wall1=Off2[1]["Off2wallN"], ifcspace2=Travel1[0], wall2=Travel1[1]["Travel1wallS"])
AmenitiesF_Travel1 = create_opening(width=1, height=2, name="AmenitiesF_Travel1", origin=(64.4, 3.0, 0), ifcspace1=AmenitiesF[0], wall1=AmenitiesF[1]["AmenitiesFwallN"], ifcspace2=Travel1[0], wall2=Travel1[1]["Travel1wallS"])

Travel1_AmenitiesM = create_opening(width=1, height=2, name="Travel1_AmenitiesM", origin=(64.4, 5.4, 0), ifcspace1=Travel1[0], wall1=Travel1[1]["Travel1wallN"], ifcspace2=AmenitiesM[0], wall2=AmenitiesM[1]["AmenitiesMwallS"])
Travel1_LiftLobby1 = create_opening(width=2, height=2, name="Travel1_LiftLobby1", origin=(45.7, 5.4, 0), ifcspace1=Travel1[0], wall1=Travel1[1]["Travel1wallN"], ifcspace2=LiftLobby1[0], wall2=LiftLobby1[1]["LiftLobby1wallS"])
Travel1_LiftLobby2 = create_opening(width=1, height=2, name="Travel1_LiftLobby2", origin=(59.7, 5.4, 0), ifcspace1=Travel1[0], wall1=Travel1[1]["Travel1wallN"], ifcspace2=LiftLobby2[0], wall2=LiftLobby2[1]["LiftLobby2wallS"])

LiftLobby1_Lift_Circ = create_opening(width=2, height=2, name="LiftLobby1_Lift_Circ", origin=(45.7, 14.1, 0), ifcspace1=LiftLobby1[0], wall1=LiftLobby1[1]["LiftLobby1wallN"], ifcspace2=Lift_Circ[0], wall2=Lift_Circ[1]["Lift_CircwallS"])
WC_Lift_Circ = create_opening(width=1, height=2, name="WC_Lift_Circ", origin=(45.1, 19.4, 0), ifcspace1=WC[0], wall1=WC[1]["WCwallE"], ifcspace2=Lift_Circ[0], wall2=Lift_Circ[1]["Lift_CircwallW"])

Lift_Circ_NE_Circ = create_opening(width=1, height=2, name="Lift_Circ_NE_Circ",origin=(46.2, 23.0, 0), ifcspace1=Lift_Circ[0], wall1=Lift_Circ[1]["Lift_CircwallN"], ifcspace2=NE_Circ[0], wall2=NE_Circ[1]["NE_CircwallS"])
NW_Circ_NE_Circ = create_opening(width=2, height=2, name="NW_Circ_NE_Circ",origin=(45.1, 23.6, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallE"], ifcspace2=NE_Circ[0], wall2=NE_Circ[1]["NE_CircwallW"])
Wait_NE_Circ = create_opening(width=1, height=2, name="Wait_NE_Circ",origin=(59.7, 23.0, 0), ifcspace1=Wait[0], wall1=Wait[1]["WaitwallN"], ifcspace2=NE_Circ[0], wall2=NE_Circ[1]["NE_CircwallS"])

LiftLobby2_Wait = create_opening(width=1, height=2, name="LiftLobby2_Wait", origin=(59.7, 16.2, 0), ifcspace1=LiftLobby2[0], wall1=LiftLobby2[1]["LiftLobby2wallN"], ifcspace2=Wait[0], wall2=Wait[1]["WaitwallS"])

ST_Gen_NW_Circ = create_opening(width=1, height=2, name="ST_Gen_NW_Circ", origin=(3.9, 23.4, 0), ifcspace1=ST_Gen[0], wall1=ST_Gen[1]["ST_GenwallE"], ifcspace2=NW_Circ[0], wall2=NW_Circ[1]["NW_CircwallW"])
Meet1_Stair_Circ = create_opening(width=1, height=2, name="Meet1_Stair_Circ", origin=(3.9, 26.5, 0), ifcspace1=Meet1[0], wall1=Meet1[1]["Meet1wallE"], ifcspace2=Stair_Circ[0], wall2=Stair_Circ[1]["Stair_CircwallW"])
StaffR_Stair_Circ = create_opening(width=1, height=2, name="StaffR_Stair_Circ", origin=(3.9, 30.0, 0), ifcspace1=StaffR[0], wall1=StaffR[1]["StaffRwallE"], ifcspace2=Stair_Circ[0], wall2=Stair_Circ[1]["Stair_CircwallW"])

NW_Circ_Stair_Circ = create_opening(width=1, height=2, name="NW_Circ_Stair_Circ", origin=(4.4, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=Stair_Circ[0], wall2=Stair_Circ[1]["Stair_CircwallS"])
NW_Circ_ExB4 = create_opening(width=2, height=2, name="NW_Circ_ExB4", origin=(10.5, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=ExB4[0], wall2=ExB4[1]["ExB4wallS"])
NW_Circ_AnBay1 = create_opening(width=2, height=2, name="NW_Circ_AnBay1", origin=(14.5, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=AnBay1[0], wall2=AnBay1[1]["AnBay1wallS"])
NW_Circ_Setup2 = create_opening(width=1, height=2, name="NW_Circ_Setup2", origin=(18.4, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=Setup2[0], wall2=Setup2[1]["Setup2wallS"])
NW_Circ_BayCirc1 = create_opening(width=1, height=2, name="NW_Circ_BayCirc1", origin=(23.8, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=BayCirc1[0], wall2=BayCirc1[1]["BayCirc1wallS"])
NW_Circ_ST_Equip2 = create_opening(width=1, height=2, name="NW_Circ_ST_Equip2", origin=(27.7, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=ST_Equip2[0], wall2=ST_Equip2[1]["ST_Equip2wallS"])
NW_Circ_ST_Equip3 = create_opening(width=1, height=2, name="NW_Circ_ST_Equip3", origin=(36.8, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=ST_Equip3[0], wall2=ST_Equip3[1]["ST_Equip3wallS"])
NW_Circ_Bay_Res = create_opening(width=2, height=2, name="NW_Circ_Bay_Res", origin=(42.0, 26.1, 0), ifcspace1=NW_Circ[0], wall1=NW_Circ[1]["NW_CircwallN"], ifcspace2=Bay_Res[0], wall2=Bay_Res[1]["Bay_ReswallS"])

ExB4_OR3 = create_opening(width=2, height=2, name="ExB4_OR3", origin=(10.5, 30.4, 0), ifcspace1=ExB4[0], wall1=ExB4[1]["ExB4wallN"], ifcspace2=OR3[0], wall2=OR3[1]["OR3wallS"])
AnBay1_OR3 = create_opening(width=2, height=2, name="AnBay1_OR3", origin=(14.5, 30.4, 0), ifcspace1=AnBay1[0], wall1=AnBay1[1]["AnBay1wallN"], ifcspace2=OR3[0], wall2=OR3[1]["OR3wallS"])
Setup2_OR3 = create_opening(width=1, height=2, name="Setup2_OR3", origin=(18.4, 30.4, 0), ifcspace1=Setup2[0], wall1=Setup2[1]["Setup2wallN"], ifcspace2=OR3[0], wall2=OR3[1]["OR3wallS"])

Clean1_BayCirc1 = create_opening(width=1, height=2, name="Clean1_BayCirc1", origin=(23.2, 30.8, 0), ifcspace1=Clean1[0], wall1=Clean1[1]["Clean1wallE"], ifcspace2=BayCirc1[0], wall2=BayCirc1[1]["BayCirc1wallW"])
BayCirc1_Dirty1 = create_opening(width=1, height=2, name="BayCirc1_Dirty1", origin=(23.4, 32.8, 0), ifcspace1=BayCirc1[0], wall1=BayCirc1[1]["BayCirc1wallN"], ifcspace2=Dirty1[0], wall2=Dirty1[1]["Dirty1wallS"])

ST_Equip3_RecBay_Circ1 = create_opening(width=1, height=2, name="ST_Equip3_RecBay_Circ1", origin=(35.8, 30.4, 0), ifcspace1=ST_Equip3[0], wall1=ST_Equip3[1]["ST_Equip3wallN"], ifcspace2=RecBay_Circ1[0], wall2=RecBay_Circ1[1]["RecBay_Circ1wallS"])
Clean3_RecBay_Circ1 = create_opening(width=1, height=2, name="Clean3_RecBay_Circ1", origin=(39.0, 30.4, 0), ifcspace1=Clean3[0], wall1=Clean3[1]["Clean3wallN"], ifcspace2=RecBay_Circ1[0], wall2=RecBay_Circ1[1]["RecBay_Circ1wallS"])
Bay_Res_RecBay_Circ1 = create_opening(width=2, height=2, name="Bay_Res_RecBay_Circ1", origin=(42.0, 30.4, 0), ifcspace1=Bay_Res[0], wall1=Bay_Res[1]["Bay_ReswallN"], ifcspace2=RecBay_Circ1[0], wall2=RecBay_Circ1[1]["RecBay_Circ1wallS"])
Dirty2_RecBay_Circ1 = create_opening(width=1, height=2, name="Dirty2_RecBay_Circ1", origin=(59.0, 30.4, 0), ifcspace1=Dirty2[0], wall1=Dirty2[1]["Dirty2wallN"], ifcspace2=RecBay_Circ1[0], wall2=RecBay_Circ1[1]["RecBay_Circ1wallS"])

RecBay_Circ1_RecBay_Circ2 = create_opening(width=2, height=2, name="RecBay_Circ1_RecBay_Circ2", origin=(65.3, 30.8, 0), ifcspace1=RecBay_Circ1[0], wall1=RecBay_Circ1[1]["RecBay_Circ1wallE"], ifcspace2=RecBay_Circ2[0], wall2=RecBay_Circ2[1]["RecBay_Circ2wallW"])
ACC1_RecBay_Circ2 = create_opening(width=1, height=2, name="ACC1_RecBay_Circ2", origin=(68.4, 30.4, 0), ifcspace1=ACC1[0], wall1=ACC1[1]["ACC1wallN"], ifcspace2=RecBay_Circ2[0], wall2=RecBay_Circ2[1]["RecBay_Circ2wallS"])

LiftLobby2_Travel2 = create_opening(width=2, height=2, name="LiftLobby2_Travel2", origin=(61.5, 14.0, 0), ifcspace1=LiftLobby2[0], wall1=LiftLobby2[1]["LiftLobby2wallE"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallW"])
Int1_Travel2 = create_opening(width=1, height=2, name="Int1_Travel2", origin=(61.7, 13.8, 0), ifcspace1=Int1[0], wall1=Int1[1]["Int1wallN"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallS"])
Int2_Travel2 = create_opening(width=1, height=2, name="Int2_Travel2", origin=(64.4, 13.8, 0), ifcspace1=Int2[0], wall1=Int2[1]["Int2wallN"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallS"])
Int3_Travel2 = create_opening(width=1, height=2, name="Int3_Travel2", origin=(67.1, 13.8, 0), ifcspace1=Int3[0], wall1=Int3[1]["Int3wallN"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallS"])
Int4_Travel2 = create_opening(width=1, height=2, name="Int4_Travel2", origin=(70.0, 13.8, 0), ifcspace1=Int4[0], wall1=Int4[1]["Int4wallN"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallS"])
Cons1_Travel2 = create_opening(width=1, height=2, name="Cons1_Travel2", origin=(74.8, 13.8, 0), ifcspace1=Cons1[0], wall1=Cons1[1]["Cons1wallN"], ifcspace2=Travel2[0], wall2=Travel2[1]["Travel2wallS"])

Travel2_Copy1 = create_opening(width=1, height=2, name="Travel2_Copy1", origin=(63.0, 16.2, 0), ifcspace1=Travel2[0], wall1=Travel2[1]["Travel2wallN"], ifcspace2=Copy1[0], wall2=Copy1[1]["Copy1wallS"])
Travel2_Change_Corr = create_opening(width=1, height=2, name="Travel2_Change_Corr", origin=(68.2, 16.2, 0), ifcspace1=Travel2[0], wall1=Travel2[1]["Travel2wallN"], ifcspace2=Change_Corr[0], wall2=Change_Corr[1]["Change_CorrwallS"])
Travel2_Lounge = create_opening(width=1, height=2, name="Travel2_Lounge", origin=(74.8, 16.2, 0), ifcspace1=Travel2[0], wall1=Travel2[1]["Travel2wallN"], ifcspace2=Lounge[0], wall2=Lounge[1]["LoungewallS"])

Lounge_RecBay_Circ3 = create_opening(width=1, height=2, name="Lounge_RecBay_Circ3", origin=(74.8, 21.7, 0), ifcspace1=Lounge[0], wall1=Lounge[1]["LoungewallN"], ifcspace2=RecBay_Circ3[0], wall2=RecBay_Circ3[1]["RecBay_Circ3wallS"])
NE_Circ_RecBay_Circ3 = create_opening(width=1, height=2, name="NE_Circ_RecBay_Circ3", origin=(70.1, 24.0, 0), ifcspace1=NE_Circ[0], wall1=NE_Circ[1]["NE_CircwallE"], ifcspace2=RecBay_Circ3[0], wall2=RecBay_Circ3[1]["RecBay_Circ3wallW"])

RecBay_Circ3_Plant = create_opening(width=2, height=2, name="RecBay_Circ3_Plant", origin=(90.1, 23.0, 0), ifcspace1=RecBay_Circ3[0], wall1=RecBay_Circ3[1]["RecBay_Circ3wallE"], ifcspace2=Plant[0], wall2=Plant[1]["PlantwallW"])
RecBay_Circ3_RecBay_Circ2 = create_opening(width=13, height=2, name="RecBay_Circ3_RecBay_Circ2", origin=(73.1, 30.4, 0), ifcspace1=RecBay_Circ3[0], wall1=RecBay_Circ3[1]["RecBay_Circ3wallN"], ifcspace2=RecBay_Circ2[0], wall2=RecBay_Circ2[1]["RecBay_Circ2wallS"])


# Exits
Stair1_Exit = create_opening(width=1, height=2, name="Stair1_Exit", origin=(4.4, 31.1, 0), ifcspace1=Stair_Circ[0], wall1=Stair_Circ[1]["Stair_CircwallN"])
Stair2_Exit = create_opening(width=1, height=2, name="Stair2_Exit", origin=(90.1, 31.0, 0), ifcspace1=RecBay_Circ2[0], wall1=RecBay_Circ2[1]["RecBay_Circ2wallE"])
Stair3_Exit = create_opening(width=1, height=2, name="Stair3_Exit", origin=(57.3, 21.6, 0), ifcspace1=Wait[0], wall1=Wait[1]["WaitwallW"])


model.write("Function_Model_hospital.ifc")

In [ ]:
import networkx as nx
import plotly.graph_objects as go

def build_nx_space_graph(model):
    """Build a graph of IfcSpaces connected through IfcOpeningElements."""
    G = nx.Graph()

    for space in model.by_type("IfcSpace"):
        s1 = space.Name or f"IfcSpace_{space.id()}"
        G.add_node(s1, ifc_id=space.id())

        for b in getattr(space, "BoundedBy", []):
            if not b.is_a("IfcRelSpaceBoundary2ndLevel"):
                continue

            cb = getattr(b, "CorrespondingBoundary", None)
            if not cb:
                continue

            elem = getattr(b, "RelatedBuildingElement", None)
            if not elem or not elem.is_a("IfcOpeningElement"):
                continue

            s2_obj = cb.RelatingSpace
            s2 = s2_obj.Name or f"IfcSpace_{s2_obj.id()}"
            opening_name = elem.Name or f"IfcOpeningElement_{elem.id()}"

            if not G.has_edge(s1, s2):
                G.add_edge(
                    s1,
                    s2,
                    opening=opening_name,
                    opening_id=elem.id(),
                    boundary_id=b.id(),
                    corresponding_boundary_id=cb.id(),
                )

    return G

# this part is just for visuals. The above is for functionality
def export_interactive_space_graph_html(model, output_html="space_graph.html"):
    G = build_nx_space_graph(model)

    if G.number_of_nodes() == 0:
        print("No IfcSpace connections found.")
        return None

    pos = nx.spring_layout(G, seed=7, k=1.8)

    # edges
    edge_x = []
    edge_y = []
    edge_hover_x = []
    edge_hover_y = []
    edge_hover_text = []

    for u, v, d in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]

        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

        mx = (x0 + x1) / 2
        my = (y0 + y1) / 2
        edge_hover_x.append(mx)
        edge_hover_y.append(my)
        edge_hover_text.append(
            f"Opening: {d['opening']}<br>"
            f"Opening ID: {d['opening_id']}<br>"
            f"Boundary ID: {d['boundary_id']}<br>"
            f"Corresponding Boundary ID: {d['corresponding_boundary_id']}"
        )

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=3, color="#1d3557"),
        hoverinfo="skip",
        name="Connections"
    )

    # invisible hover points for edges
    edge_hover_trace = go.Scatter(
        x=edge_hover_x,
        y=edge_hover_y,
        mode="markers",
        marker=dict(size=14, color="rgba(0,0,0,0)"),
        hoverinfo="text",
        text=edge_hover_text,
        name="Opening info"
    )

    # nodes
    node_x = []
    node_y = []
    node_text = []
    node_hover = []
    node_size = []

    for n, data in G.nodes(data=True):
        x, y = pos[n]
        degree = G.degree(n)

        node_x.append(x)
        node_y.append(y)
        node_text.append(n)
        node_size.append(24 + degree * 4)

        neighbors = list(G.neighbors(n))
        node_hover.append(
            f"Space: {n}<br>"
            f"Ifc ID: {data['ifc_id']}<br>"
            f"Connections: {degree}<br>"
            f"Adjacent to: {', '.join(neighbors) if neighbors else 'None'}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_text,
        textposition="top center",
        hoverinfo="text",
        hovertext=node_hover,
        marker=dict(
            size=node_size,
            color="#8ecae6",
            line=dict(width=2, color="black")
        ),
        name="Spaces"
    )

    fig = go.Figure(data=[edge_trace, edge_hover_trace, node_trace])

    fig.update_layout(
        title="IfcSpace connections through openings",
        hovermode="closest",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )

    fig.write_html(output_html, include_plotlyjs=True)
    print(f"Interactive graph exported to: {output_html}")
    print(f"Rooms: {G.number_of_nodes()}, Connections: {G.number_of_edges()}")

    return G, fig


# usage
G, fig = export_interactive_space_graph_html(model, "hospital_graph.html")